## Downloads

In [ ]:
# offline installation
!pip install -Uq /kaggle/input/library-off-for-llm/transformers-4.38.2-py3-none-any.whl
!pip install -Uq /kaggle/input/library-off-for-llm/bitsandbytes-0.42.0-py3-none-any.whl
!pip install -Uq /kaggle/input/library-off-for-llm/datasets-2.17.1-py3-none-any.whl
!pip install -Uq /kaggle/input/llm-whls/peft-0.4.0-py3-none-any.whl
!pip install -Uq /kaggle/input/llm-whls/trl-0.5.0-py3-none-any.whl

In [ ]:
import os
import random
from tqdm import tqdm
from IPython.display import display, Markdown

# pytorch
import torch

# huggingface
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig
from trl import SFTTrainer

# datasets
import pandas as pd
from datasets import Dataset

In [ ]:
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

## Device

In [ ]:
device = (
    "cuda:0" if torch.cuda.is_available() else # Nvidia GPU
    "mps" if torch.backends.mps.is_available() else # Apple Silicon GPU
    "cpu"
)
print(f"Device = {device}")

## Hyperparameters

In [ ]:
# seed
seed=42
random.seed(seed)
torch.manual_seed(seed)

# Tokenizer arguments
max_length=512
padding = "max_length"
truncation = True

# model arguments
max_new_tokens=200

# mixed precision
dtype=torch.bfloat16

# quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# LoRA configuration
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "fc1", "fc2"],
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none"
)

# training arguments
training_args = TrainingArguments(
    output_dir="./results",
    logging_dir="./logs",
    save_strategy="epoch",
    logging_strategy="steps",
    evaluation_strategy="steps",
    logging_steps=1,
    save_total_limit=1,
    report_to="none",
    
    learning_rate=2e-4,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=seed
)

# train-validation split
validation_size=0.1

# SFTTrainer arguments
max_seq_length=512

## Model

In [ ]:
# Model List

# gemma variants
#model_path = "/kaggle/input/gemma/transformers/2b-it/2"
model_path = "/kaggle/input/gemma/transformers/7b-it/2"

# llama2 variants

# phi variants
#model_path = "/kaggle/input/phi/transformers/2/1"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map=device,
    torch_dtype=dtype,
    quantization_config=quantization_config,
    trust_remote_code=True,
    use_auth_token=True
)

In [ ]:
model

## Dataset

In [ ]:
# Dataset Path
train_dataset_path1 = "/kaggle/input/llm-prompt-recovery-synthetic-datastore/gemma1000_w7b.csv"
train_dataset_path2 = "/kaggle/input/3000-rewritten-texts-prompt-recovery-challenge/prompts_0_500_wiki_first_para_3000.csv"
test_dataset_path = "/kaggle/input/llm-prompt-recovery/test.csv"

In [ ]:
# `LLM Prompt Recovery - Synthetic Datastore dataset` by @dschettler8845
df1 = pd.read_csv(train_dataset_path1)
df1 = df1[["original_text", "rewrite_prompt", "gemma_7b_rewritten_text_temp0"]]
df1 = df1.rename(columns={"gemma_7b_rewritten_text_temp0":"rewritten_text"})
df1.head()

In [ ]:
# `3000 Rewritten texts - Prompt recovery Challenge` by @dipamc77
df2 = pd.read_csv(train_dataset_path2)
df2.head()

In [ ]:
# Merge all datasets
df = pd.concat([df1, df2], axis=0)
#df = df.sample(2000).reset_index(drop=True) # to reduce training time we are only using 2k samples
print(df.shape)
df.head()

## Prompt Engineering

In [ ]:
template = """
Instruction:\n
Below, the `Original Text` passage has been rewritten into `Rewritten Text` by the `Gemma 7b-it` LLM with a certain prompt. Your task is to carefully analyze the differences between the `Original Text` and `Rewritten Text`, and try to infer the specific prompt that was likely given to the LLM to rewrite the text in this way. Start your response by writing the prompt directly. Your response should include the prompt only.\n\n

Original Text:\n
{original_text}\n\n

Rewritten Text:\n
{rewritten_text}\n\n

Response:\n
{rewrite_prompt}
"""

In [ ]:
def format_prompt(row):
    original_text = row.get("original_text", "")
    rewritten_text = row.get("rewritten_text", "")
    rewrite_prompt = row.get("rewrite_prompt", "")
    
    return template.format(
        original_text=original_text,
        rewritten_text=rewritten_text,
        rewrite_prompt=rewrite_prompt
    )

df["prompt"] = df.apply(format_prompt, axis=1)
data = df.prompt.tolist()

## Preprocessing

In [ ]:
# Convert the DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(df)

def preprocess_function(examples):
    # Tokenize the prompts
    return tokenizer(examples['prompt'], max_length=max_length, padding=padding, truncation=truncation)

# Preprocess the dataset
dataset = dataset.map(preprocess_function, batched=True)

In [ ]:
# Split the dataset into a training set and a validation set
dataset = dataset.train_test_split(test_size=validation_size, seed=seed)

# Get the training and validation sets
train_dataset = dataset['train']
val_dataset = dataset['test']

## Sample

In [ ]:
def colorize_text(text):
    for word, color in zip(["Instruction", "Original Text", "Rewritten Text", "Response"],
                           ["red", "yellow", "blue", "green"]):
        text = text.replace(f"{word}:", f"\n\n**<font color='{color}'>{word}:</font>**")
    return text

In [ ]:
# Take a random sample
sample = data[10]

# Give colors to Instruction, Response and Category
sample = colorize_text(sample)

# Show sample in markdown
display(Markdown(sample))

## Inference before Fine-Tuning

In [ ]:
def generate_response(prompt):
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    outputs = model.generate(input_ids=input_ids.to(model.device), max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Take one sample
row = df.iloc[10]

# Generate Prompt using template
prompt = template.format(
    original_text=row.original_text,
    rewritten_text=row.rewritten_text,
    rewrite_prompt="",
)

# Infer
output = generate_response(prompt)

# Colorize
output = colorize_text(output)

# Display in markdown
display(Markdown(output))

In [ ]:
# Take one sample
row = df.iloc[20]

# Generate Prompt using template
prompt = template.format(
    original_text=row.original_text,
    rewritten_text=row.rewritten_text,
    rewrite_prompt="",
)

# Infer
output = generate_response(prompt)

# Colorize
output = colorize_text(output)

# Display in markdown
display(Markdown(output))

## Supervised Fine-Tuning (LoRA)

In [ ]:
def formatting_func(examples):
    text = f"{examples['prompt'][0]}\n"
    return [text]

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    peft_config=lora_config,
    max_seq_length=max_seq_length,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    formatting_func=formatting_func
)

In [ ]:
trainer.train()

## Inference after Fine-Tuning

In [ ]:
# Take one sample
row = df.iloc[10]

# Generate Prompt using template
prompt = template.format(
    original_text=row.original_text,
    rewritten_text=row.rewritten_text,
    rewrite_prompt="",
)

# Infer
output = generate_response(prompt)

# Colorize
output = colorize_text(output)

# Display in markdown
display(Markdown(output))

In [ ]:
# Take one sample
row = df.iloc[20]

# Generate Prompt using template
prompt = template.format(
    original_text=row.original_text,
    rewritten_text=row.rewritten_text,
    rewrite_prompt="",
)

# Infer
output = generate_response(prompt)

# Colorize
output = colorize_text(output)

# Display in markdown
display(Markdown(output))

## Inference on Test Data

In [ ]:
test_df = pd.read_csv(test_dataset_path)
test_df['original_text'] = test_df['original_text'].fillna("")
test_df['rewritten_text'] = test_df['rewritten_text'].fillna("")
test_df.head()

In [ ]:
# Test Data: Take one sample
row = test_df.iloc[0]

# Generate Prompt using template
prompt = template.format(
    original_text=row.original_text,
    rewritten_text=row.rewritten_text,
    rewrite_prompt="",
)

# Infer
output = generate_response(prompt)

# Colorize
output = colorize_text(output)

# Display in markdown
display(Markdown(output))

## Save Model

In [ ]:
tokenizer.save_pretrained("waktaverse")
model.save_pretrained("waktaverse")

## Submission

In [ ]:
preds = []
for i in tqdm(range(len(test_df))):
    row = test_df.iloc[i]

    # Generate Prompt using template
    prompt = template.format(
        original_text=row.original_text,
        rewritten_text=row.rewritten_text,
        rewrite_prompt=""
    )

    # Infer
    output = generate_response(prompt)
    pred = output.replace(prompt, "") # remove the prompt from output
    
    # Store predictions
    preds.append([row.id, pred])

In [ ]:
sub_df = pd.DataFrame(preds, columns=["id", "rewrite_prompt"])
sub_df['rewrite_prompt'] = sub_df['rewrite_prompt'].fillna("")
sub_df['rewrite_prompt'] = sub_df['rewrite_prompt'].map(lambda x: "Improve the essay" if len(x) == 0 else x)
sub_df.to_csv("submission.csv",index=False)
sub_df.head()